# Similaridade entre documentos e versões

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/04_similaridade_documentos_e_versoes.ipynb)

O Notebook 03 produziu conjuntos de tokens, vetores de contagem e pesos TF-IDF.
Agora veremos que “semelhante” não é propriedade única do texto: cada medida
responde a uma relação diferente.

## 1. Escolher a medida pela pergunta

![Jaccard compara conjuntos, cosseno compara vetores, distância de edição compara sequências e a leitura próxima examina resultados divergentes.](https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/imagens/04_escolha_metrica.svg?raw=1)

| Pergunta | Representação | Medida inicial |
|---|---|---|
| Compartilham vocabulário? | conjunto de termos | Jaccard |
| Têm perfis de frequência parecidos? | vetor de contagens | cosseno |
| Compartilham termos distintivos? | vetor TF-IDF | cosseno |
| Quanto uma versão precisa mudar? | sequência de caracteres ou tokens | edição |

Autores, períodos e coleções podem ser comparados agregando documentos, desde
que autoria e pertencimento estejam documentados. **Diferente do exemplo do
notebook, nossa base tem autoria documentada** — o campo `presidente` — e três
presidentes têm mais de um discurso no corpus (Lula: 2003, 2007 e 2023; FHC:
1995 e 1999; Dilma: 2011 e 2015). Usaremos isso na Seção 6, com o cuidado que o
parágrafo acima recomenda: mostrar o número, não presumir a interpretação.

In [1]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_05'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Ambiente local: nenhuma clonagem necessária.


In [2]:
import math
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

docs = pd.read_csv("dados/derivados/documentos_processaveis.csv").sort_values("ano")

def remover_acentos(texto):
    normalizado = unicodedata.normalize("NFKD", texto)
    return "".join(ch for ch in normalizado if not unicodedata.combining(ch))

STOPWORDS_ACENTUADAS = {
    "a", "e", "o", "de", "do", "da", "dos", "das", "em", "no", "na", "nos", "nas",
    "um", "uma", "que", "com", "por", "para", "se", "ao", "aos", "à", "às", "não",
    "sua", "seu", "suas", "seus", "mais", "como", "mas", "ou", "nossa", "nosso",
    "nossas", "nossos", "esta", "este", "isso", "sem", "ser", "foi", "já",
    "também", "é", "os", "as", "ele", "ela", "eles", "elas", "lhe", "meu",
    "minha", "meus", "minhas",
}
stopwords = {remover_acentos(s) for s in STOPWORDS_ACENTUADAS}

def tokenizar(texto):
    normalizado = remover_acentos(texto.lower())
    return [t for t in re.findall(r"[a-z]+", normalizado) if t not in stopwords]

# mesma regra do Notebook 03: documento inteiro, mesma tokenização
docs["texto"] = [open(f"dados/brutos/{a}", encoding="utf-8").read() for a in docs["arquivo_bruto"]]
docs["tokens"] = docs["texto"].map(tokenizar)

matriz_contagens = pd.DataFrame(
    [Counter(t) for t in docs["tokens"]], index=docs["id_doc"]
).fillna(0).astype(int)
N = len(matriz_contagens)
df = matriz_contagens.gt(0).sum()
idf = ((N + 1) / (df + 1)).map(math.log) + 1
matriz_tfidf = matriz_contagens.mul(idf, axis=1)
print("matriz:", matriz_contagens.shape[0], "discursos ×", matriz_contagens.shape[1], "termos")

matriz: 12 discursos × 6240 termos


Com as representações reconstruídas, começamos por ignorar frequências e
comparar somente presença e ausência de termos.

## 2. Similaridade de Jaccard

Para conjuntos de termos $A$ e $B$:

$$
J(A,B)=\frac{|A\cap B|}{|A\cup B|}.
$$

O resultado varia de 0 a 1. Repetir um termo não altera Jaccard nesta definição;
dois textos de tamanhos muito diferentes podem ser penalizados pela união ampla.

In [3]:
def jaccard(tokens_a, tokens_b):
    a, b = set(tokens_a), set(tokens_b)
    return len(a & b) / len(a | b) if a | b else 1.0

consulta = "DISCURSO_2023_LULA"
tokens_consulta = docs.set_index("id_doc").loc[consulta, "tokens"]
similaridades_jaccard = docs.set_index("id_doc")["tokens"].map(
    lambda tokens: jaccard(tokens_consulta, tokens)
).drop(consulta).sort_values(ascending=False)
similaridades_jaccard.head(6).rename("Jaccard")

id_doc
DISCURSO_2011_DILMA    0.192013
DISCURSO_2007_LULA     0.189034
DISCURSO_2015_DILMA    0.188421
DISCURSO_2003_LULA     0.184904
DISCURSO_1999_FHC      0.167845
DISCURSO_2016_TEMER    0.162113
Name: Jaccard, dtype: float64

**Documento-consulta:** `DISCURSO_2023_LULA` — é o discurso mais recente do
corpus e o único cuja íntegra chegou por upload direto, não pela mesma fonte
aberta dos outros 11; comparar sua semelhança com o restante ajuda a checar se
ele se comporta como um discurso de posse "normal" no espaço vetorial, apesar
da origem documental diferente.

Jaccard trata todos os termos presentes igualmente. Para preservar frequências
ou pesos, comparamos o ângulo entre vetores.

## 3. Similaridade de cosseno

$$
\cos(\mathbf{x},\mathbf{y})=
\frac{\mathbf{x}\cdot\mathbf{y}}{\|\mathbf{x}\|_2\|\mathbf{y}\|_2}.
$$

Em vetores não negativos, 1 indica mesma direção e 0 ausência de componentes
compartilhados. Cosseno reduz o efeito da magnitude total, mas continua
dependente do vocabulário, da ponderação e do pré-processamento.

In [4]:
def cosseno(x, y):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    denominador = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.dot(x, y) / denominador) if denominador else 0.0

vetor_contagem = matriz_contagens.loc[consulta]
vetor_tfidf = matriz_tfidf.loc[consulta]
similaridades_cosseno_contagem = matriz_contagens.drop(index=consulta).apply(
    lambda linha: cosseno(vetor_contagem, linha), axis=1
).sort_values(ascending=False)
similaridades_cosseno_tfidf = matriz_tfidf.drop(index=consulta).apply(
    lambda linha: cosseno(vetor_tfidf, linha), axis=1
).sort_values(ascending=False)
ranking = pd.concat([
    similaridades_jaccard.rename("Jaccard"),
    similaridades_cosseno_contagem.rename("cosseno contagens"),
    similaridades_cosseno_tfidf.rename("cosseno TF-IDF"),
], axis=1)
ranking.sort_values("cosseno TF-IDF", ascending=False).round(3)

,Jaccard,cosseno contagens,cosseno TF-IDF
id_doc,,,
DISCURSO_2015_DILMA,0.188,0.600,0.371
DISCURSO_2003_LULA,0.185,0.567,0.336
DISCURSO_2011_DILMA,0.192,0.537,0.333
DISCURSO_1990_COLLOR,0.157,0.522,0.323
DISCURSO_2007_LULA,0.189,0.531,0.318
DISCURSO_1985_SARNEY,0.157,0.515,0.300
DISCURSO_1995_FHC,0.155,0.520,0.298
DISCURSO_1999_FHC,0.168,0.517,0.287
DISCURSO_2016_TEMER,0.162,0.469,0.282


Os três rankings podem divergir porque presença, repetição e raridade recebem
pesos diferentes. Essa divergência é material analítico, não um obstáculo a ser
escondido.

## 4. Identificar documentos semelhantes e inspecionar divergências

![Um documento consultado liga-se a vizinhos encontrados por Jaccard, cosseno de contagens e cosseno TF-IDF, incluindo um caso divergente.](https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/imagens/04_vizinhos_documentais.svg?raw=1)

Selecione pares que aparecem no topo de várias métricas e pares cuja posição
muda. Leia texto, metadados e extensão antes de chamá-los semelhantes. Nossos
metadados não têm `genero`/`local`/`tema`; usamos os campos que a base
realmente documenta: `ano`, `presidente`, `tipo_discurso`, `palavras` e
`ocorrencias_liberdade`.

In [5]:
ids_inspecao = list(dict.fromkeys(
    [consulta]
    + similaridades_jaccard.head(2).index.tolist()
    + similaridades_cosseno_tfidf.head(2).index.tolist()
))
pares_inspecao = docs.set_index("id_doc").loc[
    ids_inspecao, ["ano", "presidente", "tipo_discurso", "palavras", "ocorrencias_liberdade"]
]
display(ranking.loc[[i for i in ids_inspecao if i != consulta]].round(3))
pares_inspecao

,Jaccard,cosseno contagens,cosseno TF-IDF
id_doc,,,
DISCURSO_2011_DILMA,0.192,0.537,0.333
DISCURSO_2007_LULA,0.189,0.531,0.318
DISCURSO_2015_DILMA,0.188,0.600,0.371
DISCURSO_2003_LULA,0.185,0.567,0.336


,ano,presidente,tipo_discurso,palavras,ocorrencias_liberdade
id_doc,,,,,
DISCURSO_2023_LULA,2023,Luiz Inácio Lula da Silva,posse no Congresso Nacional,2633,7
DISCURSO_2011_DILMA,2011,Dilma Rousseff,posse no Congresso Nacional,3828,4
DISCURSO_2007_LULA,2007,Luiz Inácio Lula da Silva,posse no Congresso Nacional,3833,0
DISCURSO_2015_DILMA,2015,Dilma Rousseff,posse no Congresso Nacional,4523,3
DISCURSO_2003_LULA,2003,Luiz Inácio Lula da Silva,posse no Congresso Nacional,3881,0


Vetores são adequados para perfis lexicais. Para comparar versões quase iguais,
a ordem das unidades é crucial; passamos então à distância de edição.

## 5. Distância de edição e comparação de versões

A distância de Levenshtein é o menor número de inserções, exclusões e
substituições necessário para transformar uma sequência em outra:

$$
D_{i,j}=\min\begin{cases}
D_{i-1,j}+1 & \text{exclusão}\\
D_{i,j-1}+1 & \text{inserção}\\
D_{i-1,j-1}+\mathbf{1}(a_i\neq b_j) & \text{substituição}.
\end{cases}
$$

![Duas sequências curtas são alinhadas por operações de manter, substituir, inserir e excluir; a figura alerta que distância não interpreta o sentido.](https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/imagens/04_distancia_edicao.svg?raw=1)

A unidade pode ser caractere ou token. A distância bruta cresce com o tamanho;
uma versão normalizada divide pelo maior comprimento, com convenção declarada.
Nossa base não tem um `dados/versoes_textuais.csv` de rascunho/versão final —
mas tem um par de versões genuíno, produzido pelo próprio pipeline da Unidade 3
e documentado em `log_transformacoes.csv`: o texto **bruto** (arquivo original)
e o texto **normalizado** (NFC, `\r\n`→`\n`, espaços colapsados), recalculado
aqui com a mesma função `normalizar()` do Notebook 02 da Unidade 3.

In [6]:
def levenshtein(seq_a, seq_b):
    anterior = list(range(len(seq_b) + 1))
    for i, a in enumerate(seq_a, 1):
        atual = [i]
        for j, b in enumerate(seq_b, 1):
            atual.append(min(
                anterior[j] + 1,
                atual[j-1] + 1,
                anterior[j-1] + (a != b),
            ))
        anterior = atual
    return anterior[-1]

def normalizar(texto):
    texto = unicodedata.normalize("NFC", texto)
    texto = texto.replace("\r\n", "\n").replace("\r", "\n")
    texto = re.sub(r"[ \t\u00a0]+", " ", texto)
    return texto.strip()

resultados_edicao = []
for linha in docs.itertuples():
    bruto = linha.texto
    normalizado = normalizar(bruto)
    tokens_bruto, tokens_normalizado = tokenizar(bruto), tokenizar(normalizado)
    resultados_edicao.append({
        "id_doc": linha.id_doc,
        "idêntico caractere a caractere": bruto == normalizado,
        "distância em tokens": levenshtein(tokens_bruto, tokens_normalizado),
    })
tabela_edicao = pd.DataFrame(resultados_edicao)
tabela_edicao

,id_doc,idêntico caractere a caractere,distância em tokens
0,DISCURSO_1985_SARNEY,False,0
1,DISCURSO_1990_COLLOR,False,0
2,DISCURSO_1992_ITAMAR,False,0
3,DISCURSO_1995_FHC,False,0
4,DISCURSO_1999_FHC,False,0
5,DISCURSO_2003_LULA,False,0
6,DISCURSO_2007_LULA,False,0
7,DISCURSO_2011_DILMA,True,0
8,DISCURSO_2015_DILMA,True,0
9,DISCURSO_2016_TEMER,True,0


A distância em **tokens** é zero nos 12 discursos, mesmo nos 8 em que o texto
bruto e o normalizado diferem caractere a caractere: o tokenizador (`re.findall(r"[a-z]+")`
sobre texto sem acentos) já absorve espaços duplicados e quebras de linha — a
mesma unidade usada em todo o resto da Unidade 5 é insensível a essa edição.
Isso não significa que a normalização não fez nada: significa que, **na unidade
de análise em que comparamos documentos aqui**, ela não fez diferença. Para ver
o que de fato mudou, é preciso descer ao nível do caractere.

In [7]:
# distância de edição em nível de caractere, no menor discurso do corpus
# (Levenshtein em caracteres cresce como len(a)×len(b); nos discursos mais longos
# o cálculo levaria minutos com este algoritmo didático em Python puro)
linha = docs.set_index("id_doc").loc["DISCURSO_1992_ITAMAR"]
bruto_itamar = linha["texto"]
normalizado_itamar = normalizar(bruto_itamar)
distancia_char = levenshtein(bruto_itamar, normalizado_itamar)
pd.Series({
    "caracteres no bruto": len(bruto_itamar),
    "caracteres no normalizado": len(normalizado_itamar),
    "distância de edição (caracteres)": distancia_char,
    "distância normalizada": round(distancia_char / max(len(bruto_itamar), len(normalizado_itamar)), 4),
})

caracteres no bruto                 3681.0000
caracteres no normalizado           3670.0000
distância de edição (caracteres)      21.0000
distância normalizada                  0.0057
dtype: float64

In [8]:
import difflib

sm = difflib.SequenceMatcher(None, bruto_itamar, normalizado_itamar, autojunk=False)
operacoes = [op for op in sm.get_opcodes() if op[0] != "equal"]
print(f"{len(operacoes)} trechos não idênticos entre bruto e normalizado")
for tag, i1, i2, j1, j2 in operacoes[:6]:
    print(f"  {tag}: {bruto_itamar[i1:i2]!r} -> {normalizado_itamar[j1:j2]!r}")

11 trechos não idênticos entre bruto e normalizado
  delete: ' ' -> ''
  replace: 'É' -> 'É'
  replace: 'ú' -> 'ú'
  replace: 'ú' -> 'ú'
  replace: 'É' -> 'É'
  replace: 'ú' -> 'ú'


Os trechos divergentes são pares de caracteres visualmente idênticos (por
exemplo `ú` contra `ú`): o arquivo bruto tem alguns acentos em forma Unicode
decomposta (letra + acento como dois códigos separados) e a normalização NFC os
recompõe em um único código. **Zero mudança de sentido, zero mudança de token —
só uma convenção de codificação diferente**, exatamente o tipo de "esforço
mínimo de transformação" que o texto do notebook avisa não ser gravidade
editorial.

A distância informa esforço mínimo de transformação, não gravidade editorial ou
mudança de sentido. Corrigir um nome próprio pode custar uma substituição e ser
substantivamente decisivo.

## 6. Comparar autores, períodos e coleções

Para grupos textuais, agregue somente após decidir se a unidade é documento,
autor ou coleção. Médias de vetores por período respondem a perfis médios; unir
todos os textos responde a um megadocumento e dá mais peso a grupos maiores.
Autoria não deve ser tratada como essência estilística: gênero, período,
transcrição e composição do corpus podem explicar parte da proximidade.

Como a Seção 1 registrou, `presidente` está documentado para os 12 discursos,
e três presidentes aparecem mais de uma vez. Comparamos a similaridade média de
cosseno TF-IDF entre discursos do **mesmo** presidente contra discursos de
presidentes **diferentes** — sem presumir de antemão que a autoria explica a
proximidade, como o parágrafo acima recomenda.

In [9]:
import itertools

docs_idx = docs.set_index("id_doc")
presidentes_repetidos = docs_idx["presidente"].value_counts()
presidentes_repetidos = presidentes_repetidos[presidentes_repetidos.gt(1)]
display(presidentes_repetidos.rename("discursos no corpus"))

pares_mesmo_presidente, pares_presidentes_diferentes = [], []
for a, b in itertools.combinations(docs_idx.index, 2):
    similaridade = cosseno(matriz_tfidf.loc[a], matriz_tfidf.loc[b])
    if docs_idx.loc[a, "presidente"] == docs_idx.loc[b, "presidente"]:
        pares_mesmo_presidente.append(similaridade)
    else:
        pares_presidentes_diferentes.append(similaridade)

pd.DataFrame([
    {"grupo": "mesmo presidente", "n_pares": len(pares_mesmo_presidente),
     "cosseno TF-IDF médio": np.mean(pares_mesmo_presidente)},
    {"grupo": "presidentes diferentes", "n_pares": len(pares_presidentes_diferentes),
     "cosseno TF-IDF médio": np.mean(pares_presidentes_diferentes)},
]).round(3)

presidente
Luiz Inácio Lula da Silva    3
Fernando Henrique Cardoso    2
Dilma Rousseff               2
Name: discursos no corpus, dtype: int64

,grupo,n_pares,cosseno TF-IDF médio
0,mesmo presidente,5,0.365
1,presidentes diferentes,61,0.303


Cosseno médio um pouco mais alto entre discursos do mesmo presidente (0,365,
5 pares) do que entre presidentes diferentes (0,303, 61 pares) — mas com apenas
5 pares intra-autor (Lula tem 3 discursos = 3 pares; FHC e Dilma, 1 par cada),
essa diferença não sustenta a leitura de "estilo pessoal estável": no caso de
Lula, dois dos três discursos (2003 e 2007) têm zero ocorrências de "liberdade"
e o terceiro (2023) tem sete — o próprio termo-alvo do projeto varia dentro do
mesmo autor mais do que a semelhança vetorial global sugere.

## U05-A05 — Atividade integrada — relatório de pares

**Modalidade:** trios. **Tempo:** 35 minutos.

1. escolha um documento-consulta e três candidatos;
2. calcule Jaccard e cosseno com duas ponderações;
3. identifique uma concordância e uma divergência entre rankings;
4. leia os documentos e metadados;
5. para versões, calcule distância de edição e descreva as mudanças;
6. justifique qual medida responde melhor à pergunta.

**Pares, métricas e resultados**

- **Consulta:** `DISCURSO_2023_LULA` (n = 11 comparações).
- **Jaccard, top 2:** Dilma 2011 (0,192) e Lula 2007 (0,189) — conjuntos de
  vocabulário mais sobrepostos, sem considerar frequência.
- **Cosseno TF-IDF, top 2:** Dilma 2015 (0,371) e Lula 2003 (0,336) — perfis de
  termos distintivos mais próximos.
- **Cosseno de contagens, top 1:** Dilma 2015 (0,600), consistente com o TF-IDF.
- **Menor semelhança em todas as métricas:** Itamar 1992 (Jaccard 0,079,
  cosseno TF-IDF 0,159) — o discurso mais curto do corpus (349 tokens de
  conteúdo), o que restringe a sobreposição possível de vocabulário.

**Leitura próxima e escolha da medida**

- **Concordância:** Lula 2007 e Dilma 2011 aparecem entre os mais próximos de
  2023 em praticamente todas as métricas — os três são discursos de um mesmo
  campo político-eleitoral, em tamanho parecido (2007 e 2011 têm 3.833 e 3.804
  palavras, 2023 tem 2.633).
- **Divergência:** Jaccard prefere Dilma 2011 e Lula 2007; cosseno TF-IDF
  prefere Dilma 2015 e Lula 2003. A troca de posição faz sentido: Jaccard só
  vê presença/ausência de termos, e discursos de tamanhos parecidos tendem a
  compartilhar mais vocabulário bruto; o TF-IDF pondera pela raridade, e
  puxa para cima pares que compartilham termos **incomuns** no corpus, não
  só termos frequentes.
- **Escolha:** para a pergunta do projeto — em que discursos o vocabulário de
  "liberdade" se parece — o cosseno TF-IDF é mais adequado, porque despondera
  palavras universais (`país`, `governo`) que dominariam o Jaccard e a
  contagem bruta sem informar nada sobre o campo semântico específico do termo.

**O que a similaridade não demonstra**

O par mais próximo por cosseno TF-IDF (Dilma 2015 e Lula 2003, 0,336) não
significa que os dois tratam "liberdade" da mesma forma: Lula 2003 tem **zero**
ocorrências do termo, e Dilma 2015 tem 3. A similaridade vetorial é sobre o
perfil lexical do discurso inteiro, não sobre o conceito que a pesquisa
investiga — dois discursos podem ser vetorialmente parecidos por
compartilharem vocabulário protocolar comum (`Congresso`, `Constituição`,
`povo`) e ainda assim divergirem totalmente no uso do termo-alvo. A comparação
por autoria (Seção 6b) reforça o mesmo ponto: mesmo dentro dos três discursos
de Lula, a taxa de "liberdade" varia de 0 a 2,66/mil — autoria não é essência
estilística estável, nem para o próprio termo da pesquisa.

Leve o relatório à oficina. A entrega final exigirá uma medida principal, uma
alternativa de sensibilidade e casos que alterem ou qualifiquem a conclusão.